In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import numpy as np
from torchvision.models import mobilenet_v3_small

In [3]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, hidden, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, kernel_size=1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg = self.mlp(self.avg_pool(x))
        mx = self.mlp(self.max_pool(x))
        return x * self.sigmoid(avg + mx)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        attention = torch.cat([avg, mx], dim=1)
        attention = self.sigmoid(self.conv(attention))
        return x * attention

class CBAM(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channel = ChannelAttention(channels)
        self.spatial = SpatialAttention()
    def forward(self, x):
        x = self.channel(x)
        x = self.spatial(x)
        return x

class FeatureAlign(nn.Module):
    def __init__(self, in_channels, out_channels=128):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.Hardswish(inplace=True)
        )
    def forward(self, x, target_size):
        x = self.proj(x)
        if x.shape[-2:] != target_size:
            x = F.interpolate(x, size=target_size, mode="bilinear", align_corners=False)
        return x

class MultiLevelFusion(nn.Module):
    def __init__(self, channels=(40, 40, 96), fusion_channels=128):
        super().__init__()
        self.align1 = FeatureAlign(channels[0], fusion_channels)
        self.align2 = FeatureAlign(channels[1], fusion_channels)
        self.align3 = FeatureAlign(channels[2], fusion_channels)
        self.fuse = nn.Sequential(
            nn.Conv2d(fusion_channels * 3, fusion_channels * 2, kernel_size=1, bias=False),
            nn.BatchNorm2d(fusion_channels * 2),
            nn.Hardswish(inplace=True),
            nn.Conv2d(fusion_channels * 2, fusion_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(fusion_channels),
            nn.Hardswish(inplace=True)
        )
        self.cbam = CBAM(fusion_channels)
        self.residual = nn.Sequential(
            nn.Conv2d(fusion_channels * 3, fusion_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(fusion_channels)
        )
    def forward(self, features):
        f1, f2, f3 = features
        target_size = f1.shape[-2:]
        a1 = self.align1(f1, target_size)
        a2 = self.align2(f2, target_size)
        a3 = self.align3(f3, target_size)
        merged = torch.cat([a1, a2, a3], dim=1)
        identity = self.residual(merged)
        x = self.fuse(merged)
        x = self.cbam(x)
        return F.relu(x + identity, inplace=True)

class SpatialPyramidPooling(nn.Module):
    def __init__(self, channels, pool_sizes=(1, 2, 4)):
        super().__init__()
        self.pool_sizes = pool_sizes
    def forward(self, x):
        outputs = []
        for size in self.pool_sizes:
            pooled = F.adaptive_avg_pool2d(x, (size, size))
            outputs.append(pooled.flatten(1))
        return torch.cat(outputs, dim=1)

class StandardInceptionBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        b1, b3 = out_channels // 3, out_channels // 3
        b5 = out_channels - b1 - b3
        self.branch1 = nn.Sequential(nn.Conv2d(in_channels, b1, 1, bias=False), nn.BatchNorm2d(b1), nn.Hardswish(inplace=True))
        self.branch3 = nn.Sequential(nn.Conv2d(in_channels, b3, 3, padding=1, bias=False), nn.BatchNorm2d(b3), nn.Hardswish(inplace=True))
        self.branch5 = nn.Sequential(nn.Conv2d(in_channels, b5, 5, padding=2, bias=False), nn.BatchNorm2d(b5), nn.Hardswish(inplace=True))
    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch3(x), self.branch5(x)], dim=1)

class InceptionSEResidual(nn.Module):
    def __init__(self, channels, ratio=8):
        super().__init__()
        self.inception = StandardInceptionBlock(channels, channels)
        reduced = max(channels // ratio, 1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.se = nn.Sequential(
            nn.Conv2d(channels, reduced, 1), nn.ReLU(inplace=True),
            nn.Conv2d(reduced, channels, 1), nn.Sigmoid()
        )
    def forward(self, x):
        identity = x
        y = self.inception(x)
        scale = self.se(self.pool(y))
        return F.hardswish(y * scale + identity, inplace=True)

class ModelDV4(nn.Module):
    def __init__(self, pretrained=False, num_classes=3):
        super().__init__()
        backbone = mobilenet_v3_small(weights=None) # We don't need pretrained here since we load our weights
        
        # Convert RGB first layer to grayscale
        first_block = backbone.features[0]
        old_conv = first_block[0]
        new_conv = nn.Conv2d(1, old_conv.out_channels, old_conv.kernel_size, old_conv.stride, old_conv.padding, bias=False)
        first_block[0] = new_conv
        self.features = backbone.features

        self.inception_40_a = InceptionSEResidual(40)
        self.inception_40_b = InceptionSEResidual(40)
        self.inception_96 = InceptionSEResidual(96)
        self.fusion = MultiLevelFusion(channels=(40, 40, 96), fusion_channels=128)
        self.spp = SpatialPyramidPooling(channels=128, pool_sizes=(1, 2, 4))
        self.classifier = nn.Sequential(
            nn.Linear(128 * 21, 1024), nn.Hardswish(inplace=True),
            nn.Dropout(0.50), nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        f1 = f2 = f3 = None
        for i, layer in enumerate(self.features):
            x = layer(x)
            if i == 4: f1 = self.inception_40_a(x)
            elif i == 6: f2 = self.inception_40_b(x)
            elif i == 9: f3 = self.inception_96(x)
        fused = self.fusion([f1, f2, f3])
        pooled = self.spp(fused)
        return self.classifier(pooled)

# ==========================================
# 2. Preprocessing Function (Exact match)
# ==========================================
def preprocess_image(filepath, img_height=160, img_width=160):
    img = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
    if img is None: raise ValueError(f"Could not read image: {filepath}")

    # Square padding
    h, w = img.shape
    side = max(h, w)
    square = np.zeros((side, side), dtype=img.dtype)
    y = (side - h) // 2
    x = (side - w) // 2
    square[y:y+h, x:x+w] = img

    # Resize
    img = cv2.resize(square, (img_width, img_height), interpolation=cv2.INTER_AREA)

    # CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = clahe.apply(img)

    # Normalize to [-1, 1]
    img = img.astype(np.float32) / 255.0
    img = (img - 0.5) / 0.5
    
    # Convert to tensor (1, 1, 160, 160)
    tensor = torch.from_numpy(img).unsqueeze(0).unsqueeze(0) 
    return tensor

# ==========================================
# 3. Load Model & Run Inference
# ==========================================
def predict_image(image_path, model_path, device="cuda"):
    DEVICE = torch.device(device if torch.cuda.is_available() else "cpu")
    CLASS_NAMES = ["Corrected", "Normal", "Reversal"]

    # 1. Load Model
    model = ModelDV4(pretrained=False, num_classes=3)
    
    # Load the weights (use map_location to handle CPU/GPU mismatch)
    checkpoint = torch.load(model_path, map_location=DEVICE)
    
    # The notebook saved a dictionary with 'model_state_dict'
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        model.load_state_dict(checkpoint) # Fallback if it's just the state dict
        
    model.to(DEVICE)
    model.eval()

    # 2. Preprocess Image
    image_tensor = preprocess_image(image_path).to(DEVICE)

    # 3. Predict
    with torch.no_grad():
        logits = model(image_tensor)
        probabilities = F.softmax(logits, dim=1)[0] # Get probabilities for the batch item
        predicted_class_idx = torch.argmax(probabilities).item()

    # 4. Format Output
    predicted_class = CLASS_NAMES[predicted_class_idx]
    confidence = probabilities[predicted_class_idx].item()

    print(f"Image: {image_path}")
    print(f"Predicted Class: {predicted_class}")
    print(f"Confidence: {confidence * 100:.2f}%")
    print(f"Probabilities: Corrected: {probabilities[0]*100:.1f}% | Normal: {probabilities[1]*100:.1f}% | Reversal: {probabilities[2]*100:.1f}%")
    
    return predicted_class, confidence


In [4]:
# ==========================================
# 4. Execute
# ==========================================
if __name__ == "__main__":
    # Replace these paths with your actual paths
    MODEL_PATH = r"D:\dyslexia\model_d_v4_best.pth.zip"
    NEW_IMAGE_PATH = r"C:\Users\HP\Downloads\images.png"

    predict_image(NEW_IMAGE_PATH, MODEL_PATH)

Image: C:\Users\HP\Downloads\images.png
Predicted Class: Corrected
Confidence: 78.29%
Probabilities: Corrected: 78.3% | Normal: 8.3% | Reversal: 13.4%
